# Route A — Object Detection with YOLO
## Student notebook, Sessions 1–9

Run the cells in order. When you see `____`, that is a gap for you to fill in.

---
# Session 1 — Meet the AI that sees

**Predict:** if you had to teach a robot to find the water bottle in this picture,
what would you tell it?

### Teacher code — run this, do not study it

In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO("yolo26n.pt")
print("Model is ready.")


### Run — find things in a picture

In [ ]:
picture = cv2.imread("desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    class_id = int(box.cls[0])
    name = model.names[class_id]
    print(name)

### Investigate — how many things does it know?

In Term 4 our fruit classifier knew **2** things, and we typed in **4** rows of data by hand.

How many does this one know?

In [ ]:
print(len(model.names))
print(model.names)

### Make
Photograph three objects around the room and save each one as `my_photo.jpg`. Run the code on each one.
Write down what worked and what did not.

In [ ]:
picture = cv2.imread("my_photo.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    print(model.names[int(box.cls[0])])

---
# Session 2 — How sure is it?

**Predict:** last lesson the model got some things wrong.
Did it say nothing, or did it say something *quietly*?

### Run — print the confidence too

In [ ]:
picture = cv2.imread("desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(name, round(conf, 2))

### Investigate

In Term 4, three neighbours voted:

- all three said "apple" — we were **sure**
- two said apple, one said watermelon — we were **less sure**

Confidence is that same feeling, written as a decimal between 0 and 1.

### Modify — only show the ones it is sure about

In [ ]:
picture = cv2.imread("desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    conf = float(box.conf[0])

    if conf > 0.9:
        print(name, round(conf, 2))

### Make
Find one object that scores above 0.8, and one that scores below 0.4.
Write one sentence about why the model was less sure.

The code below uses `conf=0.1`. That tells the model to also report its quiet guesses.

In [ ]:
picture = cv2.imread("my_photo.jpg")
results = model(picture, conf=0.1, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(name, round(conf, 2))

---
# Session 3 — Where is it?

**Predict:** here is a box drawn on a picture.
Give me **four numbers** that describe where it is.

### Run — print the corners

In [ ]:
picture = cv2.imread("desk.jpg")
results = model(picture, verbose=False)

box = results[0].boxes[0]

x1 = int(box.xyxy[0][0])
y1 = int(box.xyxy[0][1])
x2 = int(box.xyxy[0][2])
y2 = int(box.xyxy[0][3])

print(x1, y1, x2, y2)

### Investigate

In Term 2, `cv2.boundingRect` gave us `x, y, w, h` — **one corner and a size**.

YOLO gives us `x1, y1, x2, y2` — **two corners**.

Same box. Different description.

**Your turn.** Do not look this up. Work out how to get `w` from `x1` and `x2`.

### Make — paper task, laptops closed
Five printed frames. Read off the coordinates by hand and work out
the width, the height and the centre for each one.

### Modify — width and centre, back on the laptop
Now turn your paper answer into code and check it against the box above.

In [ ]:
w = ____
h = ____
center_x = ____

print("width:", w)
print("height:", h)
print("centre x:", center_x)

---
# Session 4 — Many things at once

**Predict:** the table has one bottle on it. Now I add three more objects.
What changes about what the code gets back?

### Run — loop through everything it found

In [ ]:
picture = cv2.imread("messy_desk.jpg")
results = model(picture, verbose=False)

print("Things found:", len(results[0].boxes))

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(name, round(conf, 2))

### Investigate

Every vision function we wrote in Term 2 and Term 3 ended like this:

```python
return cv2.boundingRect(max(contours, key=cv2.contourArea))
```

Exactly **one** object. Always.

This one is different. Sometimes zero. Sometimes eleven.

What happens if it finds **zero** things, and we ask for the first one? Run the cell below and find out.

In [ ]:
picture = cv2.imread("blank_wall.jpg")
results = model(picture, verbose=False)

box = results[0].boxes[0]
print(model.names[int(box.cls[0])])

### Modify — check before you look

In [ ]:
picture = cv2.imread("blank_wall.jpg")
results = model(picture, verbose=False)

# Check first: did it find anything at all?
if ____:
    print("I cannot see anything.")
else:
    box = results[0].boxes[0]
    print(model.names[int(box.cls[0])])

### Make — object census
Use the messy desk photo. Count how many of each thing it finds.

`.count()` is back from Term 4, Session 1.

In [ ]:
picture = cv2.imread("messy_desk.jpg")
results = model(picture, verbose=False)

names = []
for box in results[0].boxes:
    names.append(model.names[int(box.cls[0])])

print(names)
print("bottles:", names.count("bottle"))
print("cups:", names.count("cup"))

---
# Session 5 — The right name, and sure enough

**Predict:** the model says "bottle" at 0.31 and "bottle" at 0.88.
Should the robot believe both?

### Run — only the ones with the right name

In [ ]:
picture = cv2.imread("messy_desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    conf = float(box.conf[0])

    if name == "bottle":
        print(name, round(conf, 2))

### Investigate

Two filters have to stack:

1. the right **name**
2. enough **confidence**

Neither one on its own is enough. A wrong guess at 0.3 confidence is still a
wrong guess, and a correct guess about the wrong object is still the wrong object.

`and` is back from Level 3: both sides must be true.

### Modify — stack both filters

In [ ]:
picture = cv2.imread("messy_desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    conf = float(box.conf[0])

    if name == "bottle" and ____:
        print(name, round(conf, 2))

### Make — a list of things it is sure about

Write a function that takes a picture and gives back a **list of names** the model is sure about
(confidence above 0.5). The robot will use this function in the capstone.

`imgsz=320` makes the model look at a smaller copy of the picture, which is faster.
We test that trade-off in Session 6.

In [ ]:
def confident_names(frame):
    results = model(frame, imgsz=320, verbose=False)

    names = []
    for box in results[0].boxes:
        name = model.names[int(box.cls[0])]
        conf = float(box.conf[0])

        if ____:
            names.append(____)

    return names

### Check it works

In [ ]:
picture = cv2.imread("messy_desk.jpg")
names = confident_names(picture)

print(names)
print("sure bottles:", names.count("bottle"))

---
# Session 6 — Choosing one, then going live

**Predict:** we can now find every bottle the model is sure about.
If there are three of them, which one should the robot drive to?

### Investigate — keep the best one so far

In Term 4 you found the nearest passenger out of many. You walked through the list and
remembered the best one you had seen **so far**.

We do the same here, with confidence instead of distance.

### Make — the big one

This function is the deliverable of the whole first half of term.

Look at its name and what it gives back. Now look at `find_ball()` from Term 3.

In [ ]:
def find_target(frame, target_name):
    results = model(frame, imgsz=320, verbose=False)

    best_box = None
    best_conf = 0.5

    for box in results[0].boxes:
        name = model.names[int(box.cls[0])]
        conf = float(box.conf[0])

        if name == target_name and conf > ____:
            best_conf = ____
            best_box = ____

    if best_box is None:
        return None

    x1 = int(best_box.xyxy[0][0])
    y1 = int(best_box.xyxy[0][1])
    x2 = int(best_box.xyxy[0][2])
    y2 = int(best_box.xyxy[0][3])

    return x1, y1, x2 - x1, y2 - y1

### Check it works

In [ ]:
picture = cv2.imread("messy_desk.jpg")
result = find_target(picture, "bottle")

if result is None:
    print("No bottle here.")
else:
    x, y, w, h = result
    print("Found a bottle at", x, y, "size", w, h)

### Teacher code — run this, do not study it
This opens the laptop camera on Mac, Windows or Linux, asks for a 640 × 480 picture,
and waits until the camera is ready.

In [ ]:
import platform
import time
import cv2
from IPython.display import display, Image


def open_laptop_camera():
    system = platform.system()
    if system == "Darwin":
        camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
    elif system == "Windows":
        camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    else:
        camera = cv2.VideoCapture(0)

    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    for attempt in range(50):
        ok, frame = camera.read()
        if ok and frame is not None:
            return camera, frame
        time.sleep(0.1)

    camera.release()
    raise RuntimeError("The camera did not start. Check camera permission, then restart VS Code.")


print("Camera helper is ready.")

### Make — live detection on the laptop camera
The live-video pattern is from Term 4. Stop the cell with the stop button when you are done.

In [ ]:
camera, frame = open_laptop_camera()
print("Picture size:", frame.shape[1], "x", frame.shape[0])

ok, frame_jpg = cv2.imencode(".jpg", frame)
handle = display(Image(frame_jpg.tobytes()), display_id=True)

try:
    while True:
        ok, frame = camera.read()
        if not ok:
            time.sleep(0.05)
            continue

        result = find_target(frame, "bottle")

        if result:
            x, y, w, h = result
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(frame, f"bottle w:{w}", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        ok, frame_jpg = cv2.imencode(".jpg", frame)
        if ok:
            handle.update(Image(frame_jpg.tobytes()))

        time.sleep(0.05)

finally:
    camera.release()
    print("Camera closed.")

### Modify
Change `imgsz=320` inside `find_target` to `imgsz=640`. Run the function cell again, then the live cell.
Watch the speed. Then change it back. What did we gain, and what did we lose?

---
# Session 7 — Box to number

The robot still does not move today.

**Predict:** a bottle at the far left of the picture, then the centre, then the far right.
If we made a number out of the box position, what should that number do?

In [ ]:
camera, frame = open_laptop_camera()

frame_center = frame.shape[1] // 2
target_w = 110

print("Picture width:", frame.shape[1], " centre:", frame_center)

try:
    while True:
        ok, frame = camera.read()
        if not ok:
            time.sleep(0.05)
            continue

        result = find_target(frame, "bottle")

        if result is None:
            print("Show me a bottle.")
            time.sleep(0.5)
            continue

        x, y, w, h = result
        center_x = x + w // 2

        error_x = center_x - frame_center
        error_y = target_w - w

        print(f"centre:{center_x}  error_x:{error_x}  error_y:{error_y}")
        time.sleep(0.2)

finally:
    camera.release()
    print("Camera closed.")

### Investigate — the reveal

Term 3, colour tracking:

```python
error_x = center_x - frame_center
```

Today, YOLO:

```python
error_x = center_x - frame_center
```

They are the same line.

The controller never learned anything new. We only changed **where the box comes from**.

---
# Session 8 — The robot chases

Now we swap `find_target` into the tracker you already built in Term 3.
Lines marked `# NEW` are the only changes. Everything else is the Term 3 tracker.

### Connect to the robot

In [ ]:
from ugot import ugot

got = ugot.UGOT()
got.initialize("192.168.1.___")   # type the IP address shown on your robot

### Target sizes
How wide (in pixels) each object looks when the robot is close enough to stop.
A bottle is thin and a book is wide, so they need different numbers.

In [ ]:
target_sizes = {
    "bottle": 110,
    "cup": 130,
    "book": 200,
}

In [ ]:
def target_centralizing_approaching_Kp(target_name):
    import time
    import cv2
    import numpy as np
    from IPython.display import display, Image

    # Kp values
    Kp_x = 0.06
    Kp_y = 0.3

    target_w = target_sizes[target_name]   # NEW: each object has its own size
    dead_zone_x = 30
    dead_zone_y = 20                       # NEW: YOLO boxes wobble, so a wider dead zone
    max_lost_frames = 30                   # NEW: give up if the target disappears

    # function 1: get the bgr image
    def picture_bgr():
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            return None
        return cv2.imdecode(np.frombuffer(picture_raw, np.uint8), cv2.IMREAD_COLOR)

    # function 2: THE SWAP - this used to be find_ball()
    def find_it(frame):
        return find_target(frame, target_name)

    # function 3: Kp move and bounding box
    def move_and_draw(frame, x, y, w, h):
        frame_center = frame.shape[1] // 2   # NEW: read the centre from the picture
        center_x = x + w // 2

        error_x = center_x - frame_center
        error_y = target_w - w

        if abs(error_x) < dead_zone_x and abs(error_y) < dead_zone_y:
            got.mecanum_stop()
            return True

        speed_x = int(Kp_x * error_x)
        speed_y = int(Kp_y * error_y)

        speed_x = max(min(speed_x, 15), -15)
        speed_y = max(min(speed_y, 15), -15)

        got.mecanum_move_xyz(speed_x, speed_y, 0)

        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, f"{target_name} ex:{error_x} ey:{error_y}", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        return False

    # main program
    got.open_camera()

    frame = None                           # NEW: wait for the first picture
    for attempt in range(50):
        frame = picture_bgr()
        if frame is not None:
            break
        time.sleep(0.1)
    if frame is None:
        print("The robot camera did not start.")
        return False

    ok, picture_jpg = cv2.imencode(".jpg", frame)
    handle = display(Image(picture_jpg.tobytes()), display_id=True)

    arrived = False
    lost_frames = 0

    try:
        while True:
            frame = picture_bgr()
            if frame is None:
                time.sleep(0.05)           # NEW: do not spin at full speed
                continue

            result = find_it(frame)
            if result:
                lost_frames = 0
                if move_and_draw(frame, *result):
                    arrived = True
                    break
            else:
                got.mecanum_stop()
                lost_frames = lost_frames + 1          # NEW
                if lost_frames > max_lost_frames:      # NEW
                    print(f"I lost the {target_name}.")
                    break

            ok, picture_jpg = cv2.imencode(".jpg", frame)
            if ok:
                handle.update(Image(picture_jpg.tobytes()))

            time.sleep(0.05)

    finally:
        got.mecanum_stop()
        print("Robot stopped.")

    return arrived

In [ ]:
arrived = target_centralizing_approaching_Kp("bottle")
print("Arrived:", arrived)

### Investigate
The robot overshoots more than it did with colour tracking. Why?

### Modify
Make `Kp_x` smaller. Make the dead zones wider. Tune it by feel.
Then try `"cup"` instead of `"bottle"`.

---
# Session 9 — Capstone: the Lost Property Robot

The robot is given a list of lost items. It looks around step by step until it spots one,
drives up to it, and announces what it found.

Each step is **sense → think → act**:

1. **Sense:** stand still and take a picture.
2. **Think:** ask `confident_names()` what it can see, then check the list.
3. **Act:** if nothing is there, turn a little and try again.

Nothing new. Just assembly.

In [ ]:
import time
import numpy as np
import cv2

lost_items = ["bottle", "cup", "book"]


def picture_bgr():
    picture_raw = got.read_camera_data()
    if picture_raw is None:
        return None
    return cv2.imdecode(np.frombuffer(picture_raw, np.uint8), cv2.IMREAD_COLOR)


def search_for_lost_item():
    got.open_camera()
    got.play_audio_tts("I am looking for lost property.", 1, True)

    try:
        for step in range(18):
            # SENSE: stand still, then take a picture
            time.sleep(0.5)
            frame = picture_bgr()

            # THINK: run the model once, then check every lost item
            if frame is not None:
                names = confident_names(frame)
                for item in lost_items:
                    if names.count(item) > 0:
                        print(f"I found a {item}!")
                        got.play_audio_tts(f"I found a {item}.", 1, True)
                        return item

            # ACT: nothing here, so turn left 20 degrees and look again
            got.mecanum_turn_speed_times(2, 30, 20, 2)
            time.sleep(0.8)

        got.play_audio_tts("I could not find any lost property.", 1, True)
        return None

    finally:
        got.mecanum_stop()

In [ ]:
found_item = search_for_lost_item()

if found_item is None:
    print("Nothing found. Put an object near the robot and try again.")
else:
    time.sleep(0.5)
    arrived = target_centralizing_approaching_Kp(found_item)

    if arrived:
        got.play_audio_tts(f"Here is your {found_item}. Job done!", 1, True)
    else:
        got.play_audio_tts(f"I lost the {found_item}. Please try again.", 1, True)

### Extension — only if the simple version already runs
Deliver the item to an AprilTag zone using `check_point_navigation()` from Term 1.